# Metaheuristics (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/metaheuristics_cpp.ipynb)

Executable notebook of the chapter “Metaheuristics” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the metaheuristics of the chapter: simulated annealing, tabu search, iterated local search, variable neighbourhood search, GRASP, large neighbourhood search and the genetic and memetic algorithm, written in C++ and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/metaheuristics/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/metaheuristics_python.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ file is written to the folder `src/` by its `%%writefile` cell; the cell after `Main.cpp` compiles everything with `g++ -std=c++17 -O2` and runs the examples. Nothing to install (Colab already has `g++`).

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Declarations

The header declares all the functions of the chapter; each function is in its own file, with the same name.

In [1]:
%%writefile src/Metaheuristics.h
#ifndef METAHEURISTICS_HEADER
#define METAHEURISTICS_HEADER

// metaheuristics for the KP, the BPP, the TSP and P||Cmax
// items, jobs and cities are indexed 0, 1, ..., n-1; machines 0, ..., m-1; bins 0, ..., nb-1
// a TSP tour is a permutation pi[0..n-1] of the cities with pi[0] = 0 (the first city)

#include <vector>
#include <cmath>
#include <cstddef>
using namespace std;

typedef vector<int> Vec;
typedef vector<Vec> Matrix;

// random numbers: Park-Miller minimal standard generator, or the numbers of the list "given"
struct Random
{
    long long seed;
    vector<double> given;
    int used;
};
double Rand(Random& r);

// TSP
int TourLength(const Vec& pi, const Matrix& d);
int TwoOptDelta(const Vec& pi, const Matrix& d, int i, int j);
void ApplyTwoOpt(Vec& pi, int i, int j);
void TwoOptLS(Vec& pi, const Matrix& d);
Vec NearestNeighbour(const Matrix& d);
int SimulatedAnnealingTSP(const Matrix& d, Vec& pi, double T0, double alpha, int L, double Tmin, Random& r,
                          Vec* curve = NULL);
int TabuSearchTSP(const Matrix& d, Vec& pi, int tenure, int maxIter);
Vec DoubleBridge(const Vec& pi, Random& r);
int IteratedLocalSearchTSP(const Matrix& d, Vec& pi, int maxIter, Random& r);
Vec RandomizedNN(const Matrix& d, double alpha, Random& r);
int GraspTSP(const Matrix& d, Vec& best, double alpha, int maxIter, Random& r);
Vec RandomRemoval(Vec& pi, int q, Random& r);
Vec WorstRemoval(Vec& pi, int q, const Matrix& d);
void CheapestInsertionRepair(Vec& pi, const Vec& removed, const Matrix& d);
int LnsTSP(const Matrix& d, Vec& pi, int q, int maxIter, Random& r);
int AlnsTSP(const Matrix& d, Vec& pi, int q, int maxIter, double rho, Random& r, double* wgt);

// P||Cmax
Vec LPT(const Vec& t, int m);
Vec Loads(const Vec& mach, const Vec& t, int m);
bool LexLess(const Vec& L1, const Vec& L2);
int TabuSearchPCmax(const Vec& t, int m, Vec& mach, int tenure, int maxIter);
void MoveDescent(const Vec& t, int m, Vec& mach);
Vec Shake(const Vec& t, int m, const Vec& mach, int k, Random& r);
int VNSPCmax(const Vec& t, int m, Vec& mach, int kmax, int maxIter, Random& r);

// KP
int Profit(const Vec& p, const Vec& x);
int Weight(const Vec& w, const Vec& x);
void KPLocalSearch(const Vec& p, const Vec& w, int c, Vec& x);
Vec RandomizedGreedyKP(const Vec& p, const Vec& w, int c, double alpha, Random& r);
int GraspKP(const Vec& p, const Vec& w, int c, Vec& best, double alpha, int maxIter, Random& r);
int SimulatedAnnealingKP(const Vec& p, const Vec& w, int c, Vec& x, int beta,
                         double T0, double alpha, int L, double Tmin, Random& r);
void RepairKP(const Vec& p, const Vec& w, int c, Vec& x);
int Tournament(const Vec& fit, Random& r);
int GeneticKP(const Vec& p, const Vec& w, int c, Vec& best, int P, double pm, int maxIter,
              bool memetic, Random& r);

// BPP
int FirstFit(const Vec& w, int c, const Vec& order, Vec& bin);
Vec DestroyBins(const Vec& bin, int nb, const Vec& w, int q, Random& r);
int RepairFFD(const Vec& items, const Vec& w, int c, Vec& asg);
int RepairExact(const Vec& items, const Vec& w, int c, Vec& asg);
int LnsBPP(const Vec& w, int c, Vec& bin, int nb, int q, bool exact, int maxIter, Random& r);

#endif

Writing src/Metaheuristics.h


## Random numbers

The random numbers are produced by `Rand`, the Park–Miller minimal standard generator. The structure `Random` contains the seed and, optionally, a list of given numbers, which are returned instead: the exercises give their random numbers explicitly.

In [2]:
%%writefile src/Rand.cpp
#include "Metaheuristics.h"

// Park-Miller minimal standard generator: seed = 16807 * seed mod (2^31 - 1), u = seed /
// (2^31 - 1)
// if a list of numbers is given, they are returned in order instead

double Rand(Random& r)
{
    double u;
    if (r.given.size() > 0)
    {
        u = r.given[r.used];
    }
    else
    {
        r.seed = (16807 * r.seed) % 2147483647;
        u = (double) r.seed / 2147483647.0;
    }
    r.used = r.used + 1;
    return u;
}

Writing src/Rand.cpp


## Local searches and auxiliary functions

The 2-opt local search for the TSP, with the $O(1)$ evaluation of a move; the move descent for $P\,||\,C_{\max}$ with the lexicographic comparison, and the LPT starting solution; the local search for the KP with feasible flips and swaps.

In [3]:
%%writefile src/TourLength.cpp
#include "Metaheuristics.h"

// length L(pi) of the tour pi

int TourLength(const Vec& pi, const Matrix& d)
{
    int n = pi.size();
    int len = 0;
    for (int k = 0; k < n; ++k)
    {
        len = len + d[pi[k]][pi[(k + 1) % n]];
    }
    return len;
}

Writing src/TourLength.cpp


In [4]:
%%writefile src/TwoOptDelta.cpp
#include "Metaheuristics.h"

// change of length of the 2-opt move (i,j): the edges (pi[i],pi[i+1]) and (pi[j],pi[j+1])
// are replaced by (pi[i],pi[j]) and (pi[i+1],pi[j+1]); computed in O(1)

int TwoOptDelta(const Vec& pi, const Matrix& d, int i, int j)
{
    int n = pi.size();
    int a = pi[i], b = pi[i + 1], c = pi[j], e = pi[(j + 1) % n];
    return d[a][c] + d[b][e] - d[a][b] - d[c][e];
}

Writing src/TwoOptDelta.cpp


In [5]:
%%writefile src/ApplyTwoOpt.cpp
#include "Metaheuristics.h"

// performs the 2-opt move (i,j): reverses the segment pi[i+1..j]

void ApplyTwoOpt(Vec& pi, int i, int j)
{
    int lo = i + 1, hi = j;
    while (lo < hi)
    {
        int tmp = pi[lo]; pi[lo] = pi[hi]; pi[hi] = tmp;
        lo = lo + 1; hi = hi - 1;
    }
}

Writing src/ApplyTwoOpt.cpp


In [6]:
%%writefile src/TwoOptLS.cpp
#include "Metaheuristics.h"

// 2-opt local search with best improvement: stops at a local optimum
// ties between moves are broken in favour of the first move (i,j) in lexicographic order

void TwoOptLS(Vec& pi, const Matrix& d)
{
    int n = pi.size();
    while (true)
    {
        int bdelta = 0, bi = -1, bj = -1;
        for (int i = 0; i < n - 1; ++i)
        {
            for (int j = i + 2; j < n; ++j)
            {
                if (i == 0 && j == n - 1) continue;      // same two edges: no change
                int delta = TwoOptDelta(pi, d, i, j);
                if (delta < bdelta) { bdelta = delta; bi = i; bj = j; }
            }
        }
        // no improving move: local optimum
        if (bi < 0) return;
        ApplyTwoOpt(pi, bi, bj);
    }
}

Writing src/TwoOptLS.cpp


In [7]:
%%writefile src/NearestNeighbour.cpp
#include "Metaheuristics.h"

// nearest neighbour tour from city 0 (ties: smallest index)

Vec NearestNeighbour(const Matrix& d)
{
    int n = d.size();
    Vec pi(1, 0);
    vector<bool> visited(n, false);
    visited[0] = true;
    for (int k = 1; k < n; ++k)
    {
        int last = pi.back(), next = -1;
        for (int j = 0; j < n; ++j)
        {
            if (!visited[j] && (next < 0 || d[last][j] < d[last][next])) next = j;
        }
        pi.push_back(next);
        visited[next] = true;
    }
    return pi;
}

Writing src/NearestNeighbour.cpp


In [8]:
%%writefile src/Loads.cpp
#include "Metaheuristics.h"

// loads L[h] of the machines for the assignment mach

Vec Loads(const Vec& mach, const Vec& t, int m)
{
    Vec L(m, 0);
    for (int j = 0; j < (int) t.size(); ++j) L[mach[j]] = L[mach[j]] + t[j];
    return L;
}

Writing src/Loads.cpp


In [9]:
%%writefile src/LexLess.cpp
#include "Metaheuristics.h"

// true if the loads L1 are better than L2 in the lexicographic order
// (makespan first, then sum of the squared loads)

bool LexLess(const Vec& L1, const Vec& L2)
{
    int C1 = 0, C2 = 0, S1 = 0, S2 = 0;
    for (int h = 0; h < (int) L1.size(); ++h)
    {
        if (L1[h] > C1) C1 = L1[h];
        if (L2[h] > C2) C2 = L2[h];
        S1 = S1 + L1[h] * L1[h];
        S2 = S2 + L2[h] * L2[h];
    }
    return C1 < C2 || (C1 == C2 && S1 < S2);
}

Writing src/LexLess.cpp


In [10]:
%%writefile src/MoveDescent.cpp
#include "Metaheuristics.h"

// local search for P||Cmax with the move neighbourhood and best improvement,
// with respect to the lexicographic order (makespan, sum of squared loads)

void MoveDescent(const Vec& t, int m, Vec& mach)
{
    int n = t.size();
    Vec L = Loads(mach, t, m);
    while (true)
    {
        int bj = -1, bh = -1;
        Vec bL = L;
        for (int j = 0; j < n; ++j)
        {
            for (int h = 0; h < m; ++h)
            {
                if (h == mach[j]) continue;
                Vec L2 = L;
                L2[mach[j]] = L2[mach[j]] - t[j];
                L2[h] = L2[h] + t[j];
                if (LexLess(L2, bL)) { bj = j; bh = h; bL = L2; }
            }
        }
        if (bj < 0) return;                              // local optimum
        mach[bj] = bh;
        L = bL;
    }
}

Writing src/MoveDescent.cpp


In [11]:
%%writefile src/LPT.cpp
#include "Metaheuristics.h"

// longest processing time first: jobs by non-increasing t (ties: smallest index),
// each assigned to the least loaded machine (ties: smallest index)

Vec LPT(const Vec& t, int m)
{
    int n = t.size();
    Vec order(n), mach(n), L(m, 0);
    for (int j = 0; j < n; ++j) order[j] = j;
    for (int a = 1; a < n; ++a)                          // insertion sort, stable
    {
        int j = order[a], b = a;
        while (b > 0 && t[order[b - 1]] < t[j]) { order[b] = order[b - 1]; b = b - 1; }
        order[b] = j;
    }
    for (int a = 0; a < n; ++a)
    {
        int j = order[a], h = 0;
        for (int k = 1; k < m; ++k) if (L[k] < L[h]) h = k;
        mach[j] = h;
        L[h] = L[h] + t[j];
    }
    return mach;
}

Writing src/LPT.cpp


In [12]:
%%writefile src/Profit.cpp
#include "Metaheuristics.h"

// total profit of the items selected by x

int Profit(const Vec& p, const Vec& x)
{
    int P = 0;
    for (int i = 0; i < (int) p.size(); ++i) if (x[i] == 1) P = P + p[i];
    return P;
}

Writing src/Profit.cpp


In [13]:
%%writefile src/Weight.cpp
#include "Metaheuristics.h"

// total weight of the items selected by x

int Weight(const Vec& w, const Vec& x)
{
    int W = 0;
    for (int i = 0; i < (int) w.size(); ++i) if (x[i] == 1) W = W + w[i];
    return W;
}

Writing src/Weight.cpp


In [14]:
%%writefile src/KPLocalSearch.cpp
#include "Metaheuristics.h"

// local search for the KP with best improvement on the feasible neighbours of the flip
// neighbourhood (only additions can improve) and of the swap neighbourhood

void KPLocalSearch(const Vec& p, const Vec& w, int c, Vec& x)
{
    int n = p.size();
    while (true)
    {
        int W = Weight(w, x), P = Profit(p, x);
        // bi = -1: add item bk; else swap bi out, bk in
        int bP = P, bi = -1, bk = -1;
        for (int k = 0; k < n; ++k)
            if (x[k] == 0 && W + w[k] <= c && P + p[k] > bP)
            { bP = P + p[k]; bi = -1; bk = k; }
        for (int i = 0; i < n; ++i)
        {
            if (x[i] != 1) continue;
            for (int k = 0; k < n; ++k)
                if (x[k] == 0 && W - w[i] + w[k] <= c && P - p[i] + p[k] > bP)
                { bP = P - p[i] + p[k]; bi = i; bk = k; }
        }
        if (bk < 0) return;                              // local optimum
        if (bi >= 0) x[bi] = 0;
        x[bk] = 1;
    }
}

Writing src/KPLocalSearch.cpp


## Simulated annealing

Simulated annealing for the TSP, with the 2-opt neighbourhood, and for the KP with the penalized objective.

In [15]:
%%writefile src/SimulatedAnnealingTSP.cpp
#include "Metaheuristics.h"

// simulated annealing for the TSP with the 2-opt neighbourhood and geometric cooling
// pi: starting tour on input, best tour found on output; returns its length
// if curve is not NULL, the length of the current tour after each move is appended to it

int SimulatedAnnealingTSP(const Matrix& d, Vec& pi, double T0, double alpha, int L,
                          double Tmin, Random& r, Vec* curve)
{
    int n = pi.size();
    // the M = n(n-3)/2 moves (i,j) of 2-opt
    Vec mi, mj;
    for (int i = 0; i < n - 1; ++i)
        for (int j = i + 2; j < n; ++j)
            if (!(i == 0 && j == n - 1)) { mi.push_back(i); mj.push_back(j); }
    int M = mi.size();

    Vec cur = pi;                                        // current solution
    int fcur = TourLength(cur, d);
    Vec best = cur;                                      // best solution found
    int fbest = fcur;
    double T = T0;                                       // temperature
    while (T > Tmin)
    {
        for (int l = 0; l < L; ++l)                      // L moves at temperature T
        {
            int k = (int) (Rand(r) * M);                 // random neighbour
            int delta = TwoOptDelta(cur, d, mi[k], mj[k]);
            double v = Rand(r);
            if (delta <= 0 || v < exp(-delta / T))       // Metropolis acceptance rule
            {
                ApplyTwoOpt(cur, mi[k], mj[k]);
                fcur = fcur + delta;
                if (fcur < fbest) { best = cur; fbest = fcur; }
            }
            if (curve != NULL) curve->push_back(fcur);
        }
        T = alpha * T;                                   // cooling
    }
    pi = best;
    return fbest;
}

Writing src/SimulatedAnnealingTSP.cpp


In [16]:
%%writefile src/SimulatedAnnealingKP.cpp
#include "Metaheuristics.h"

// simulated annealing for the KP with the flip neighbourhood and the penalized objective
// f(x) = profit(x) - beta * max{0, weight(x) - c}; infeasible solutions may be visited,
// the best solution is updated with feasible solutions only
// x: feasible starting solution on input, best solution found on output; returns its
// profit

int SimulatedAnnealingKP(const Vec& p, const Vec& w, int c, Vec& x, int beta,
                         double T0, double alpha, int L, double Tmin, Random& r)
{
    int n = p.size();
    int excess = Weight(w, x) - c;
    int f = Profit(p, x) - beta * (excess > 0 ? excess : 0);
    Vec best = x;
    int zbest = Profit(p, x);
    double T = T0;
    while (T > Tmin)
    {
        for (int l = 0; l < L; ++l)
        {
            int k = (int) (Rand(r) * n);                 // random flip
            Vec y = x;
            y[k] = 1 - y[k];
            int ey = Weight(w, y) - c;
            int fy = Profit(p, y) - beta * (ey > 0 ? ey : 0);
            // maximization: delta > 0 is a worsening
            int delta = f - fy;
            double v = Rand(r);
            if (delta <= 0 || v < exp(-delta / T))
            {
                x = y; f = fy;
                if (ey <= 0 && Profit(p, x) > zbest) { best = x; zbest = Profit(p, x); }
            }
        }
        T = alpha * T;
    }
    x = best;
    return zbest;
}

Writing src/SimulatedAnnealingKP.cpp


## Tabu search

Tabu search for $P\,||\,C_{\max}$ and for the TSP.

In [17]:
%%writefile src/TabuSearchPCmax.cpp
#include "Metaheuristics.h"

// tabu search for P||Cmax with the move neighbourhood (one job to another machine);
// the best non-tabu move is performed even if it is worse (makespan first, then sum of
// squares);
// after moving job j away from machine h, the attribute (j,h) is tabu for tenure
// iterations;
// a tabu move is allowed if it gives a makespan smaller than the best one (aspiration)
// mach: starting assignment on input, best assignment on output; returns its makespan

int TabuSearchPCmax(const Vec& t, int m, Vec& mach, int tenure, int maxIter)
{
    int n = t.size();
    Vec L = Loads(mach, t, m);
    // tabu[j][h] >= it: j may not go to h
    Matrix tabu(n, Vec(m, 0));
    Vec best = mach;
    int bestC = 0, sum = 0, LB = 0;
    for (int h = 0; h < m; ++h) if (L[h] > bestC) bestC = L[h];
    for (int j = 0; j < n; ++j) { sum = sum + t[j]; if (t[j] > LB) LB = t[j]; }
    // LB = max{ max t_j, ceil(sum/m) }
    if ((sum + m - 1) / m > LB) LB = (sum + m - 1) / m;
    int it = 0;
    while (it < maxIter && bestC > LB)
    {
        it = it + 1;
        int bj = -1, bh = -1;
        Vec bL;
        for (int j = 0; j < n; ++j)
        {
            for (int h = 0; h < m; ++h)
            {
                if (h == mach[j]) continue;
                Vec L2 = L;
                L2[mach[j]] = L2[mach[j]] - t[j];
                L2[h] = L2[h] + t[j];
                int C2 = 0;
                for (int k = 0; k < m; ++k) if (L2[k] > C2) C2 = L2[k];
                // tabu and no aspiration
                if (tabu[j][h] >= it && !(C2 < bestC)) continue;
                if (bj < 0 || LexLess(L2, bL)) { bj = j; bh = h; bL = L2; }
            }
        }
        if (bj < 0) break;
        // j may not go back for tenure iterations
        tabu[bj][mach[bj]] = it + tenure;
        mach[bj] = bh;
        L = bL;
        int C = 0;
        for (int k = 0; k < m; ++k) if (L[k] > C) C = L[k];
        if (C < bestC) { bestC = C; best = mach; }
    }
    mach = best;
    return bestC;
}

Writing src/TabuSearchPCmax.cpp


In [18]:
%%writefile src/TabuSearchTSP.cpp
#include "Metaheuristics.h"

// tabu search for the TSP with the 2-opt neighbourhood
// the two edges removed by a move may not be added back for tenure iterations,
// unless the move gives a tour shorter than the best one (aspiration criterion)

int TabuSearchTSP(const Matrix& d, Vec& pi, int tenure, int maxIter)
{
    int n = pi.size();
    Vec cur = pi;
    int lcur = TourLength(cur, d);
    Vec best = cur;
    int lbest = lcur;
    // tabu[a][b] >= it: edge (a,b) is tabu
    Matrix tabu(n, Vec(n, 0));
    for (int it = 1; it <= maxIter; ++it)
    {
        int bdelta = 0, bi = -1, bj = -1;
        for (int i = 0; i < n - 1; ++i)
        {
            for (int j = i + 2; j < n; ++j)
            {
                if (i == 0 && j == n - 1) continue;
                int a = cur[i], b = cur[i + 1], c = cur[j], e = cur[(j + 1) % n];
                int delta = d[a][c] + d[b][e] - d[a][b] - d[c][e];
                bool isTabu = tabu[a][c] >= it || tabu[b][e] >= it;
                if (isTabu && !(lcur + delta < lbest)) continue;
                if (bi < 0 || delta < bdelta) { bdelta = delta; bi = i; bj = j; }
            }
        }
        if (bi < 0) break;                               // every move is tabu
        int a = cur[bi], b = cur[bi + 1], c = cur[bj], e = cur[(bj + 1) % n];
        tabu[a][b] = tabu[b][a] = it + tenure;           // removed edges become tabu
        tabu[c][e] = tabu[e][c] = it + tenure;
        // best non-tabu move, even if worse
        ApplyTwoOpt(cur, bi, bj);
        lcur = lcur + bdelta;
        if (lcur < lbest) { best = cur; lbest = lcur; }
    }
    pi = best;
    return lbest;
}

Writing src/TabuSearchTSP.cpp


## Iterated local search and variable neighbourhood search

Iterated local search for the TSP with the double-bridge kick, and variable neighbourhood search for $P\,||\,C_{\max}$.

In [19]:
%%writefile src/DoubleBridge.cpp
#include "Metaheuristics.h"

// double-bridge kick: the tour A B C D, cut at the positions p1 < p2 < p3, becomes A C B
// D

Vec DoubleBridge(const Vec& pi, Random& r)
{
    int n = pi.size();
    int p1 = 1 + (int) (Rand(r) * (n - 3));
    int p2 = p1 + 1 + (int) (Rand(r) * (n - 2 - p1));
    int p3 = p2 + 1 + (int) (Rand(r) * (n - 1 - p2));
    Vec kick;
    for (int k = 0; k < p1; ++k) kick.push_back(pi[k]);     // A
    for (int k = p2; k < p3; ++k) kick.push_back(pi[k]);    // C
    for (int k = p1; k < p2; ++k) kick.push_back(pi[k]);    // B
    for (int k = p3; k < n; ++k) kick.push_back(pi[k]);     // D
    return kick;
}

Writing src/DoubleBridge.cpp


In [20]:
%%writefile src/IteratedLocalSearchTSP.cpp
#include "Metaheuristics.h"

// iterated local search for the TSP: double-bridge kick + 2-opt local search,
// the new local optimum is accepted if it is better than the current one

int IteratedLocalSearchTSP(const Matrix& d, Vec& pi, int maxIter, Random& r)
{
    Vec cur = pi;
    TwoOptLS(cur, d);                                    // first local optimum
    int lcur = TourLength(cur, d);
    for (int it = 1; it <= maxIter; ++it)
    {
        Vec nw = DoubleBridge(cur, r);                   // perturbation
        TwoOptLS(nw, d);                                 // local search
        int lnew = TourLength(nw, d);
        if (lnew < lcur) { cur = nw; lcur = lnew; }      // acceptance criterion
    }
    pi = cur;
    return lcur;
}

Writing src/IteratedLocalSearchTSP.cpp


In [21]:
%%writefile src/Shake.cpp
#include "Metaheuristics.h"

// shaking in the k-th neighbourhood: k random moves, each one moving a random job j
// to a random machine different from mach[j]

Vec Shake(const Vec& t, int m, const Vec& mach, int k, Random& r)
{
    int n = t.size();
    Vec sh = mach;
    for (int s = 0; s < k; ++s)
    {
        int j = (int) (Rand(r) * n);
        int g = (int) (Rand(r) * (m - 1));
        // the (g+1)-th machine other than sh[j]
        sh[j] = (g < sh[j]) ? g : g + 1;
    }
    return sh;
}

Writing src/Shake.cpp


In [22]:
%%writefile src/VNSPCmax.cpp
#include "Metaheuristics.h"

// basic variable neighbourhood search for P||Cmax: shaking with k random moves,
// local search by MoveDescent; k returns to 1 after an improvement, otherwise it grows
// up to kmax

int VNSPCmax(const Vec& t, int m, Vec& mach, int kmax, int maxIter, Random& r)
{
    int n = t.size(), sum = 0, LB = 0;
    for (int j = 0; j < n; ++j) { sum = sum + t[j]; if (t[j] > LB) LB = t[j]; }
    if ((sum + m - 1) / m > LB) LB = (sum + m - 1) / m;
    MoveDescent(t, m, mach);
    Vec L = Loads(mach, t, m);
    int C = 0;
    for (int h = 0; h < m; ++h) if (L[h] > C) C = L[h];
    int it = 0, k = 1;
    while (it < maxIter && C > LB)
    {
        it = it + 1;
        Vec nw = Shake(t, m, mach, k, r);                // shaking
        MoveDescent(t, m, nw);                           // local search
        Vec Lnew = Loads(nw, t, m);
        if (LexLess(Lnew, L))                            // move or not
        {
            mach = nw; L = Lnew; k = 1;
            C = 0;
            for (int h = 0; h < m; ++h) if (L[h] > C) C = L[h];
        }
        else
        {
            k = (k < kmax) ? k + 1 : 1;
        }
    }
    return C;
}

Writing src/VNSPCmax.cpp


## GRASP

GRASP for the KP and for the TSP.

In [23]:
%%writefile src/RandomizedGreedyKP.cpp
#include "Metaheuristics.h"

// greedy randomized construction for the KP: at each step the candidates are the items
// that fit,
// RCL = { i candidate : p_i/w_i >= emax - alpha (emax - emin) }, sorted by non-increasing
// efficiency (ties: smallest index), and the item is chosen at random in the RCL

Vec RandomizedGreedyKP(const Vec& p, const Vec& w, int c, double alpha, Random& r)
{
    int n = p.size(), rc = c;                            // rc: residual capacity
    Vec x(n, 0);
    while (true)
    {
        double emax = -1, emin = -1;
        for (int i = 0; i < n; ++i)
        {
            if (x[i] == 1 || w[i] > rc) continue;
            double e = (double) p[i] / w[i];
            if (emax < 0 || e > emax) emax = e;
            if (emin < 0 || e < emin) emin = e;
        }
        if (emax < 0) return x;                          // no item fits
        double thr = emax - alpha * (emax - emin);
        Vec rcl;
        for (int i = 0; i < n; ++i)
        {
            if (x[i] == 1 || w[i] > rc || (double) p[i] / w[i] < thr - 1e-9) continue;
            int pos = rcl.size();                        // insertion in sorted position
            rcl.push_back(i);
            while (pos > 0 && p[rcl[pos - 1]] * w[i] < p[i] * w[rcl[pos - 1]])
            { rcl[pos] = rcl[pos - 1]; pos = pos - 1; }
            rcl[pos] = i;
        }
        int i = rcl[(int) (Rand(r) * rcl.size())];
        x[i] = 1;
        rc = rc - w[i];
    }
}

Writing src/RandomizedGreedyKP.cpp


In [24]:
%%writefile src/GraspKP.cpp
#include "Metaheuristics.h"

// GRASP for the KP: greedy randomized construction + local search, maxIter times

int GraspKP(const Vec& p, const Vec& w, int c, Vec& best, double alpha, int maxIter,
            Random& r)
{
    int zbest = -1;
    for (int it = 1; it <= maxIter; ++it)
    {
        Vec x = RandomizedGreedyKP(p, w, c, alpha, r);   // construction
        KPLocalSearch(p, w, c, x);                       // local search
        int z = Profit(p, x);
        if (z > zbest) { best = x; zbest = z; }
    }
    return zbest;
}

Writing src/GraspKP.cpp


In [25]:
%%writefile src/RandomizedNN.cpp
#include "Metaheuristics.h"

// randomized nearest neighbour from city 0: the next city is chosen at random in the
// restricted candidate list RCL = { j unvisited : d[last][j] <= dmin + alpha (dmax -
// dmin) },
// sorted by non-decreasing distance (ties: smallest index)

Vec RandomizedNN(const Matrix& d, double alpha, Random& r)
{
    int n = d.size();
    Vec pi(1, 0);
    vector<bool> visited(n, false);
    visited[0] = true;
    for (int k = 1; k < n; ++k)
    {
        int last = pi.back();
        int dmin = -1, dmax = -1;
        for (int j = 0; j < n; ++j)
        {
            if (visited[j]) continue;
            if (dmin < 0 || d[last][j] < dmin) dmin = d[last][j];
            if (dmax < 0 || d[last][j] > dmax) dmax = d[last][j];
        }
        double thr = dmin + alpha * (dmax - dmin);
        Vec rcl;
        for (int j = 0; j < n; ++j)
        {
            if (visited[j] || d[last][j] > thr + 1e-9) continue;
            int pos = rcl.size();                        // insertion in sorted position
            rcl.push_back(j);
            while (pos > 0 && d[last][rcl[pos - 1]] > d[last][j])
            { rcl[pos] = rcl[pos - 1]; pos = pos - 1; }
            rcl[pos] = j;
        }
        int next = rcl[(int) (Rand(r) * rcl.size())];
        pi.push_back(next);
        visited[next] = true;
    }
    return pi;
}

Writing src/RandomizedNN.cpp


In [26]:
%%writefile src/GraspTSP.cpp
#include "Metaheuristics.h"

// GRASP for the TSP: randomized nearest neighbour + 2-opt local search, maxIter times

int GraspTSP(const Matrix& d, Vec& best, double alpha, int maxIter, Random& r)
{
    int lbest = -1;
    for (int it = 1; it <= maxIter; ++it)
    {
        Vec pi = RandomizedNN(d, alpha, r);              // greedy randomized construction
        TwoOptLS(pi, d);                                 // local search
        int len = TourLength(pi, d);
        if (lbest < 0 || len < lbest) { best = pi; lbest = len; }
    }
    return lbest;
}

Writing src/GraspTSP.cpp


## Large neighbourhood search

LNS and ALNS for the TSP; LNS for the BPP, with the First Fit starting packing, the destroy method and the two repair methods.

In [27]:
%%writefile src/RandomRemoval.cpp
#include "Metaheuristics.h"

// destroy: removes q random cities from the tour pi (never the first city) and returns
// them

Vec RandomRemoval(Vec& pi, int q, Random& r)
{
    Vec removed;
    for (int s = 0; s < q; ++s)
    {
        int k = 1 + (int) (Rand(r) * (pi.size() - 1));
        removed.push_back(pi[k]);
        pi.erase(pi.begin() + k);
    }
    return removed;
}

Writing src/RandomRemoval.cpp


In [28]:
%%writefile src/WorstRemoval.cpp
#include "Metaheuristics.h"

// destroy: removes, q times, the city whose removal saves the most length (never the
// first city)

Vec WorstRemoval(Vec& pi, int q, const Matrix& d)
{
    Vec removed;
    for (int s = 0; s < q; ++s)
    {
        int n = pi.size(), bk = -1, bs = 0;
        for (int k = 1; k < n; ++k)
        {
            int a = pi[k - 1], v = pi[k], b = pi[(k + 1) % n];
            int saving = d[a][v] + d[v][b] - d[a][b];
            if (bk < 0 || saving > bs) { bk = k; bs = saving; }
        }
        removed.push_back(pi[bk]);
        pi.erase(pi.begin() + bk);
    }
    return removed;
}

Writing src/WorstRemoval.cpp


In [29]:
%%writefile src/CheapestInsertionRepair.cpp
#include "Metaheuristics.h"

// repair: inserts the removed cities, in the order of removal, each in the position
// of minimum insertion cost d[a][v] + d[v][b] - d[a][b] (ties: first position)

void CheapestInsertionRepair(Vec& pi, const Vec& removed, const Matrix& d)
{
    for (int s = 0; s < (int) removed.size(); ++s)
    {
        int v = removed[s], n = pi.size(), bpos = -1, bcost = 0;
        for (int k = 0; k < n; ++k)
        {
            int a = pi[k], b = pi[(k + 1) % n];
            int cost = d[a][v] + d[v][b] - d[a][b];
            if (bpos < 0 || cost < bcost) { bpos = k; bcost = cost; }
        }
        pi.insert(pi.begin() + bpos + 1, v);
    }
}

Writing src/CheapestInsertionRepair.cpp


In [30]:
%%writefile src/LnsTSP.cpp
#include "Metaheuristics.h"

// large neighbourhood search for the TSP: random removal of q cities + cheapest
// insertion,
// the new tour is accepted if it is shorter than the current one

int LnsTSP(const Matrix& d, Vec& pi, int q, int maxIter, Random& r)
{
    Vec cur = pi;
    int lcur = TourLength(cur, d);
    for (int it = 1; it <= maxIter; ++it)
    {
        Vec nw = cur;
        Vec removed = RandomRemoval(nw, q, r);           // destroy
        CheapestInsertionRepair(nw, removed, d);         // repair
        int lnew = TourLength(nw, d);
        if (lnew < lcur) { cur = nw; lcur = lnew; }      // acceptance
    }
    pi = cur;
    return lcur;
}

Writing src/LnsTSP.cpp


In [31]:
%%writefile src/AlnsTSP.cpp
#include "Metaheuristics.h"

// adaptive large neighbourhood search for the TSP with two destroy operators
// (o = 0: random removal, o = 1: worst removal) and cheapest insertion as repair;
// the operator is chosen with probability proportional to its weight wgt[o],
// updated as wgt[o] = (1 - rho) wgt[o] + rho psi, with psi = 3, 2, 1, 0 if the new tour
// is a new best, better than the current one, as good as the current one, worse

int AlnsTSP(const Matrix& d, Vec& pi, int q, int maxIter, double rho, Random& r,
            double* wgt)
{
    Vec cur = pi;
    int lcur = TourLength(cur, d);
    Vec best = cur;
    int lbest = lcur;
    wgt[0] = 1.0; wgt[1] = 1.0;
    for (int it = 1; it <= maxIter; ++it)
    {
        int o = (Rand(r) * (wgt[0] + wgt[1]) < wgt[0]) ? 0 : 1;     // roulette wheel
        Vec nw = cur;
        Vec removed = (o == 0) ? RandomRemoval(nw, q, r) : WorstRemoval(nw, q, d);
        CheapestInsertionRepair(nw, removed, d);
        int lnew = TourLength(nw, d);
        double psi;
        if (lnew < lbest) psi = 3.0;
        else if (lnew < lcur) psi = 2.0;
        else if (lnew == lcur) psi = 1.0;
        else psi = 0.0;
        if (lnew <= lcur) { cur = nw; lcur = lnew; }
        if (lcur < lbest) { best = cur; lbest = lcur; }
        wgt[o] = (1 - rho) * wgt[o] + rho * psi;                   // adaptive weights
    }
    pi = best;
    return lbest;
}

Writing src/AlnsTSP.cpp


In [32]:
%%writefile src/FirstFit.cpp
#include "Metaheuristics.h"

// first fit on the items in the given order; returns the number of bins nb, bin[i] = bin
// of item i

int FirstFit(const Vec& w, int c, const Vec& order, Vec& bin)
{
    Vec load;
    bin.assign(w.size(), -1);
    for (int a = 0; a < (int) order.size(); ++a)
    {
        int i = order[a], b = 0;
        while (b < (int) load.size() && load[b] + w[i] > c) b = b + 1;
        if (b == (int) load.size()) load.push_back(0);   // open a new bin
        bin[i] = b;
        load[b] = load[b] + w[i];
    }
    return load.size();
}

Writing src/FirstFit.cpp


In [33]:
%%writefile src/DestroyBins.cpp
#include "Metaheuristics.h"

// destroy for the BPP: the least loaded bin (ties: smallest index) and q-1 other bins at
// random

Vec DestroyBins(const Vec& bin, int nb, const Vec& w, int q, Random& r)
{
    Vec load(nb, 0);
    for (int i = 0; i < (int) w.size(); ++i) load[bin[i]] = load[bin[i]] + w[i];
    int e = 0;
    for (int b = 1; b < nb; ++b) if (load[b] < load[e]) e = b;
    Vec empty(1, e);
    while ((int) empty.size() < q && (int) empty.size() < nb)
    {
        Vec others;
        for (int b = 0; b < nb; ++b)
        {
            bool chosen = false;
            for (int s = 0; s < (int) empty.size(); ++s) if (empty[s] == b) chosen = true;
            if (!chosen) others.push_back(b);
        }
        empty.push_back(others[(int) (Rand(r) * others.size())]);
    }
    return empty;
}

Writing src/DestroyBins.cpp


In [34]:
%%writefile src/RepairFFD.cpp
#include "Metaheuristics.h"

// heuristic repair for the BPP: first fit decreasing on the removed items, in new bins;
// asg[s] = bin (0, 1, ...) of items[s]; returns the number of new bins

int RepairFFD(const Vec& items, const Vec& w, int c, Vec& asg)
{
    int k = items.size();
    Vec order(k);
    for (int s = 0; s < k; ++s) order[s] = s;
    for (int a = 1; a < k; ++a)                          // non-increasing weight, stable
    {
        int s = order[a], b = a;
        while (b > 0 && w[items[order[b - 1]]] < w[items[s]])
        { order[b] = order[b - 1]; b = b - 1; }
        order[b] = s;
    }
    Vec load;
    asg.assign(k, -1);
    for (int a = 0; a < k; ++a)
    {
        int s = order[a], b = 0;
        while (b < (int) load.size() && load[b] + w[items[s]] > c) b = b + 1;
        if (b == (int) load.size()) load.push_back(0);
        asg[s] = b;
        load[b] = load[b] + w[items[s]];
    }
    return load.size();
}

Writing src/RepairFFD.cpp


In [35]:
%%writefile src/RepairExact.cpp
#include "Metaheuristics.h"

// exact repair for the BPP: optimal packing of the removed items in new bins, by a
// depth-first
// branch and bound (items by non-increasing weight; each item goes in an open bin or in
// a new one)

static void Branch(int a, const Vec& order, const Vec& items, const Vec& w, int c,
                   Vec& load, Vec& cur, Vec& asg, int& best)
{
    if ((int) load.size() >= best) return;              // bound: cannot improve
    if (a == (int) order.size()) { best = load.size(); asg = cur; return; }
    int s = order[a];
    for (int b = 0; b < (int) load.size(); ++b)
    {
        if (load[b] + w[items[s]] > c) continue;
        load[b] = load[b] + w[items[s]]; cur[s] = b;
        Branch(a + 1, order, items, w, c, load, cur, asg, best);
        load[b] = load[b] - w[items[s]];
    }
    load.push_back(w[items[s]]); cur[s] = load.size() - 1;
    Branch(a + 1, order, items, w, c, load, cur, asg, best);
    load.pop_back();
}

int RepairExact(const Vec& items, const Vec& w, int c, Vec& asg)
{
    int k = items.size();
    Vec order(k);
    for (int s = 0; s < k; ++s) order[s] = s;
    for (int a = 1; a < k; ++a)
    {
        int s = order[a], b = a;
        while (b > 0 && w[items[order[b - 1]]] < w[items[s]])
        { order[b] = order[b - 1]; b = b - 1; }
        order[b] = s;
    }
    Vec load, cur(k, -1);
    int best = k + 1;
    Branch(0, order, items, w, c, load, cur, asg, best);
    return best;
}

Writing src/RepairExact.cpp


In [36]:
%%writefile src/LnsBPP.cpp
#include "Metaheuristics.h"

// large neighbourhood search for the BPP: empty q bins (DestroyBins) and repack their
// items
// with FFD (exact = false) or optimally (exact = true); the new packing is accepted if
// it uses
// fewer bins, or the same number of bins and a sum of squared loads not smaller
// bin, nb: starting packing on input, final packing on output; returns the number of bins

int LnsBPP(const Vec& w, int c, Vec& bin, int nb, int q, bool exact, int maxIter,
           Random& r)
{
    int n = w.size(), sum = 0;
    for (int i = 0; i < n; ++i) sum = sum + w[i];
    int LB = (sum + c - 1) / c;
    int it = 0;
    while (it < maxIter && nb > LB)
    {
        it = it + 1;
        Vec empty = DestroyBins(bin, nb, w, q, r);      // destroy
        vector<bool> isEmptied(nb, false);
        for (int s = 0; s < (int) empty.size(); ++s) isEmptied[empty[s]] = true;
        Vec items, newbin(n, -1), newIndex(nb, -1);
        int kept = 0;
        for (int b = 0; b < nb; ++b)
            if (!isEmptied[b]) { newIndex[b] = kept; kept = kept + 1; }
        for (int i = 0; i < n; ++i)
        {
            if (isEmptied[bin[i]]) items.push_back(i);
            else newbin[i] = newIndex[bin[i]];
        }
        Vec asg;                                         // repair
        int k = exact ? RepairExact(items, w, c, asg) : RepairFFD(items, w, c, asg);
        for (int s = 0; s < (int) items.size(); ++s) newbin[items[s]] = kept + asg[s];
        int nb2 = kept + k;
        Vec load(nb, 0), load2(nb2, 0);
        for (int i = 0; i < n; ++i) { load[bin[i]] += w[i]; load2[newbin[i]] += w[i]; }
        // auxiliary objective: sum of squared loads
        long long aux = 0, aux2 = 0;
        for (int b = 0; b < nb; ++b) aux = aux + load[b] * load[b];
        for (int b = 0; b < nb2; ++b) aux2 = aux2 + load2[b] * load2[b];
        // acceptance
        if (nb2 < nb || (nb2 == nb && aux2 >= aux)) { bin = newbin; nb = nb2; }
    }
    return nb;
}

Writing src/LnsBPP.cpp


## Genetic and memetic algorithm

The genetic and memetic algorithm for the KP.

In [37]:
%%writefile src/RepairKP.cpp
#include "Metaheuristics.h"

// repair of an infeasible KP solution: removes the selected item of minimum efficiency
// (ties: largest index) until the capacity is respected

void RepairKP(const Vec& p, const Vec& w, int c, Vec& x)
{
    int n = p.size(), W = Weight(w, x);
    while (W > c)
    {
        int s = -1;
        for (int i = 0; i < n; ++i)
            if (x[i] == 1 && (s < 0 || p[i] * w[s] <= p[s] * w[i])) s = i;
        x[s] = 0;
        W = W - w[s];
    }
}

Writing src/RepairKP.cpp


In [38]:
%%writefile src/Tournament.cpp
#include "Metaheuristics.h"

// binary tournament: two random individuals, the fitter one wins (ties: the first drawn)

int Tournament(const Vec& fit, Random& r)
{
    int P = fit.size();
    int a = (int) (Rand(r) * P);
    int b = (int) (Rand(r) * P);
    return (fit[a] >= fit[b]) ? a : b;
}

Writing src/Tournament.cpp


In [39]:
%%writefile src/GeneticKP.cpp
#include "Metaheuristics.h"

// steady-state genetic algorithm for the KP with binary encoding: binary tournament
// selection,
// one-point crossover, bit-flip mutation with probability pm, repair; if memetic is true,
// every new individual is also improved by KPLocalSearch; the child replaces the worst
// individual (ties: last index) if it is better and not already in the population

int GeneticKP(const Vec& p, const Vec& w, int c, Vec& best, int P, double pm, int maxIter,
              bool memetic, Random& r)
{
    int n = p.size();
    Matrix pop(P, Vec(n, 0));
    Vec fit(P, 0);
    for (int s = 0; s < P; ++s)                          // random initial population
    {
        for (int i = 0; i < n; ++i) pop[s][i] = (Rand(r) < 0.5) ? 1 : 0;
        RepairKP(p, w, c, pop[s]);
        if (memetic) KPLocalSearch(p, w, c, pop[s]);
        fit[s] = Profit(p, pop[s]);
    }
    for (int it = 1; it <= maxIter; ++it)
    {
        int a = Tournament(fit, r);                      // selection
        int b = Tournament(fit, r);
        int cut = 1 + (int) (Rand(r) * (n - 1));         // one-point crossover
        Vec child(n);
        for (int i = 0; i < n; ++i) child[i] = (i < cut) ? pop[a][i] : pop[b][i];
        for (int i = 0; i < n; ++i)                      // mutation
            if (Rand(r) < pm) child[i] = 1 - child[i];
        RepairKP(p, w, c, child);                        // repair
        if (memetic) KPLocalSearch(p, w, c, child);      // memetic: local search
        int fc = Profit(p, child);
        int worst = 0;
        for (int s = 1; s < P; ++s) if (fit[s] <= fit[worst]) worst = s;
        bool present = false;
        for (int s = 0; s < P; ++s) if (pop[s] == child) present = true;
        // replacement
        if (!present && fc > fit[worst]) { pop[worst] = child; fit[worst] = fc; }
    }
    int sb = 0;
    for (int s = 1; s < P; ++s) if (fit[s] > fit[sb]) sb = s;
    best = pop[sb];
    return fit[sb];
}

Writing src/GeneticKP.cpp


## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [40]:
%%writefile src/Main.cpp
#include "Metaheuristics.h"
#include <cstdio>
#include <cstring>

// runs every example of the lecture notes and every instance of the exercises, and compares the
// value found with the one computed by an independent implementation (generated file);
// "METAHEUR data" writes the data of the simulated annealing figure in data/sa_tsp20.dat

static int errors = 0, checks = 0;

static void check(const char* name, int found, int expected)
{
    checks++;
    if (found != expected) errors++;
    printf("%-46s %6d %s\n", name, found, found == expected ? "OK" : "<-- EXPECTED A DIFFERENT VALUE");
}

static Random S(long long seed) { Random r; r.seed = seed; r.used = 0; return r; }
static Random G(const vector<double>& g) { Random r; r.seed = 1; r.given = g; r.used = 0; return r; }

int main(int argc, char** argv)
{
    Matrix dR = Matrix{Vec{0,5,2,2,9},Vec{5,0,4,3,4},Vec{2,4,0,4,7},Vec{2,3,4,0,7},Vec{9,4,7,7,0}};
    Matrix d20 = Matrix{Vec{0,58,33,55,79,83,30,53,72,86,38,88,46,92,23,21,63,36,35,93},Vec{58,0,33,14,28,37,29,18,35,28,42,37,21,36,53,45,10,27,23,36},Vec{33,33,0,39,48,51,9,22,40,59,44,56,15,63,20,34,34,6,22,65},Vec{55,14,39,0,42,51,32,31,49,37,30,50,31,46,57,38,24,33,21,45},Vec{79,28,48,42,0,10,49,26,18,20,70,9,33,17,67,71,19,43,49,22},Vec{83,37,51,51,10,0,54,31,14,29,78,9,37,22,69,77,27,47,56,29},Vec{30,29,9,32,49,54,0,23,44,57,35,58,16,62,25,26,33,7,13,64},Vec{53,18,22,31,26,31,23,0,22,38,50,35,7,41,42,47,14,17,26,44},Vec{72,35,40,49,18,14,44,22,0,38,72,22,28,34,56,69,26,37,49,40},Vec{86,28,59,37,20,29,57,38,38,0,67,23,44,11,79,73,24,53,51,8},Vec{38,42,44,30,70,78,35,50,72,67,0,79,46,77,52,17,51,41,24,75},Vec{88,37,56,50,9,9,58,35,22,23,79,0,42,14,75,80,28,52,58,21},Vec{46,21,15,31,33,37,16,7,28,44,46,42,0,48,35,42,20,10,22,51},Vec{92,36,63,46,17,22,62,41,34,11,77,14,48,0,82,81,29,57,59,7},Vec{23,53,20,57,67,69,25,42,56,79,52,75,35,82,0,37,55,26,37,85},Vec{21,45,34,38,71,77,26,47,69,73,17,80,42,81,37,0,53,33,22,81},Vec{63,10,34,24,19,27,33,14,26,24,51,28,20,29,55,53,0,29,30,31},Vec{36,27,6,33,43,47,7,17,37,53,41,52,10,57,26,33,29,0,17,60},Vec{35,23,22,21,49,56,13,26,49,51,24,58,22,59,37,22,30,17,0,59},Vec{93,36,65,45,22,29,64,44,40,8,75,21,51,7,85,81,31,60,59,0}};
    Vec pR = Vec{10,7,6,4}, wR = Vec{5,4,3,3}; int cR = 9;
    Vec p20 = Vec{54,55,55,52,58,42,53,53,22,64,54,51,39,49,53,46,28,64,58,34}, w20 = Vec{39,43,49,47,44,35,43,44,16,50,40,42,28,38,46,39,14,50,45,27}; int c20 = 389;
    Vec tR = Vec{5,5,4,4,3,3,3}; int mR = 3;
    Vec t15 = Vec{10,23,26,16,23,11,14,18,18,24,29,22,13,13,15}; int m15 = 4;
    Vec wB = Vec{4,4,5,3,2,3}; int cB = 7;
    Vec w24 = Vec{30,40,57,22,37,23,34,34,55,51,21,47,22,24,22,34,20,51,25,54,35,27,55,48}; int c24 = 100;
    Vec nnR = NearestNeighbour(dR), nn20 = NearestNeighbour(d20);
    Vec ord24 = Vec{2,8,22,19,9,17,23,11,1,4,20,6,7,15,0,21,18,13,5,3,12,14,10,16};
    if (argc > 1 && strcmp(argv[1], "data") == 0)
    {
        Vec pi = nn20, curve;
        Random r = S(1);
        SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r, &curve);
        Vec best(curve.size());
        int b = TourLength(nn20, d20);
        for (int k = 0; k < (int) curve.size(); ++k) { if (curve[k] < b) b = curve[k]; best[k] = b; }
        FILE* f = fopen("data/sa_tsp20.dat", "w");
        fprintf(f, "it cur best\n");
        for (int k = 9; k < (int) curve.size(); k += 10) fprintf(f, "%d %d %d\n", k + 1, curve[k], best[k]);
        fclose(f);
        printf("data/sa_tsp20.dat written\n");
        return 0;
    }
    check("TSP ref: nearest neighbour", TourLength(nnR, dR), 25);
    check("Ex SA TSP ref (seed 1)", [&]{ Vec pi = nnR; Random r = S(1); return SimulatedAnnealingTSP(dR, pi, 5.0, 0.8, 2, 3.0, r); }(), 18);
    check("Ex SA KP ref (seed 3)", [&]{ Vec x = {1,0,1,0}; Random r = S(3); return SimulatedAnnealingKP(pR, wR, cR, x, 3, 4.0, 0.8, 2, 2.5, r); }(), 17);
    check("Ex TS PCmax ref tenure=0", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 0, 50); }(), 11);
    check("Ex TS PCmax ref tenure=1", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 1, 50); }(), 11);
    check("Ex TS PCmax ref tenure=2", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 2, 50); }(), 11);
    check("Ex TS PCmax ref tenure=3", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 3, 50); }(), 11);
    check("Ex TS PCmax ref tenure=4", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 4, 50); }(), 10);
    check("Ex TS PCmax ref tenure=5", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 5, 50); }(), 10);
    check("Ex TS PCmax ref tenure=6", [&]{ Vec mach = LPT(tR, mR); return TabuSearchPCmax(tR, mR, mach, 6, 50); }(), 9);
    check("Ex VNS PCmax ref (seed 1)", [&]{ Vec mach = LPT(tR, mR); Random r = S(1); return VNSPCmax(tR, mR, mach, 3, 50, r); }(), 9);
    check("Ex GRASP KP ref (seed 2)", [&]{ Vec b; Random r = S(2); return GraspKP(pR, wR, cR, b, 0.5, 2, r); }(), 17);
    check("Ex LNS TSP ref (seed 1)", [&]{ Vec pi = nnR; Random r = S(1); return LnsTSP(dR, pi, 2, 1, r); }(), 18);
    check("Ex LNS BPP ref (seed 3)", [&]{ Vec bin; int nb = FirstFit(wB, cB, Vec{0,1,2,3,4,5}, bin); Random r = S(3); return LnsBPP(wB, cB, bin, nb, 2, false, 10, r); }(), 3);
    check("Ex GA KP ref (seed 36)", [&]{ Vec b; Random r = S(36); return GeneticKP(pR, wR, cR, b, 4, 0.25, 3, false, r); }(), 17);
    check("TSP20 nearest neighbour", TourLength(nn20, d20), 326);
    check("TSP20 2-opt", [&]{ Vec pi = nn20; TwoOptLS(pi, d20); return TourLength(pi, d20); }(), 319);
    check("TSP20 SA alpha=0.9 T0=1 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 299);
    check("TSP20 SA alpha=0.9 T0=1 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.9 T0=1 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=1 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=1 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=1 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.9 T0=1 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 322);
    check("TSP20 SA alpha=0.9 T0=1 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.9 T0=1 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.9 T0=1 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=5 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=5 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=5 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=5 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=5 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=5 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=5 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=5 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=5 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=5 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.9, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.9 T0=20 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 294);
    check("TSP20 SA alpha=0.9 T0=20 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 304);
    check("TSP20 SA alpha=0.9 T0=20 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=20 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 310);
    check("TSP20 SA alpha=0.9 T0=20 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 323);
    check("TSP20 SA alpha=0.9 T0=20 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=20 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=20 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 310);
    check("TSP20 SA alpha=0.9 T0=20 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=20 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=100 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=100 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=100 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=100 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=100 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.9 T0=100 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 311);
    check("TSP20 SA alpha=0.9 T0=100 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 303);
    check("TSP20 SA alpha=0.9 T0=100 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.9 T0=100 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 299);
    check("TSP20 SA alpha=0.9 T0=100 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.9, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=1 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 299);
    check("TSP20 SA alpha=0.95 T0=1 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=1 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.95 T0=1 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=1 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.95 T0=1 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.95 T0=1 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=1 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=1 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 319);
    check("TSP20 SA alpha=0.95 T0=1 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.95, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.95 T0=5 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=5 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=5 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=5 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=5 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=5 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=5 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=5 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=5 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=5 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.95, 100, 0.5, r); }(), 296);
    check("TSP20 SA alpha=0.95 T0=20 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=20 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=20 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 296);
    check("TSP20 SA alpha=0.95 T0=20 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=20 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=20 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=20 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=20 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=20 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=20 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.95 T0=100 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 299);
    check("TSP20 SA alpha=0.95 T0=100 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.95 T0=100 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.95, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.99 T0=1 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 299);
    check("TSP20 SA alpha=0.99 T0=1 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.99 T0=1 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.99 T0=1 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=1 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.99 T0=1 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.99 T0=1 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=1 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=1 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=1 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 1.0, 0.99, 100, 0.5, r); }(), 305);
    check("TSP20 SA alpha=0.99 T0=5 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 298);
    check("TSP20 SA alpha=0.99 T0=5 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=5 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 5.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=20 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 20.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 SA alpha=0.99 T0=100 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return SimulatedAnnealingTSP(d20, pi, 100.0, 0.99, 100, 0.5, r); }(), 291);
    check("TSP20 TS tenure=0", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 0, 200); }(), 319);
    check("TSP20 TS tenure=1", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 1, 200); }(), 291);
    check("TSP20 TS tenure=2", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 2, 200); }(), 291);
    check("TSP20 TS tenure=5", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 5, 200); }(), 291);
    check("TSP20 TS tenure=10", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 10, 200); }(), 291);
    check("TSP20 TS tenure=20", [&]{ Vec pi = nn20; return TabuSearchTSP(d20, pi, 20, 200); }(), 291);
    check("TSP20 ILS seed 1", [&]{ Vec pi = nn20; Random r = S(1); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 2", [&]{ Vec pi = nn20; Random r = S(2); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 3", [&]{ Vec pi = nn20; Random r = S(3); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 4", [&]{ Vec pi = nn20; Random r = S(4); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 5", [&]{ Vec pi = nn20; Random r = S(5); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 6", [&]{ Vec pi = nn20; Random r = S(6); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 7", [&]{ Vec pi = nn20; Random r = S(7); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 8", [&]{ Vec pi = nn20; Random r = S(8); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 9", [&]{ Vec pi = nn20; Random r = S(9); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 ILS seed 10", [&]{ Vec pi = nn20; Random r = S(10); return IteratedLocalSearchTSP(d20, pi, 100, r); }(), 291);
    check("TSP20 GRASP alpha=0 seed 1", [&]{ Vec b; Random r = S(1); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 1", [&]{ Vec b; Random r = S(1); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 2", [&]{ Vec b; Random r = S(2); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 2", [&]{ Vec b; Random r = S(2); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 3", [&]{ Vec b; Random r = S(3); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 3", [&]{ Vec b; Random r = S(3); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 4", [&]{ Vec b; Random r = S(4); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 4", [&]{ Vec b; Random r = S(4); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 5", [&]{ Vec b; Random r = S(5); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 5", [&]{ Vec b; Random r = S(5); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 6", [&]{ Vec b; Random r = S(6); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 6", [&]{ Vec b; Random r = S(6); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 7", [&]{ Vec b; Random r = S(7); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 7", [&]{ Vec b; Random r = S(7); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 8", [&]{ Vec b; Random r = S(8); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 8", [&]{ Vec b; Random r = S(8); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 9", [&]{ Vec b; Random r = S(9); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 9", [&]{ Vec b; Random r = S(9); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0 seed 10", [&]{ Vec b; Random r = S(10); return GraspTSP(d20, b, 0.0, 20, r); }(), 319);
    check("KP20 GRASP alpha=0 seed 10", [&]{ Vec b; Random r = S(10); return GraspKP(p20, w20, c20, b, 0.0, 20, r); }(), 514);
    check("TSP20 GRASP alpha=0.1 seed 1", [&]{ Vec b; Random r = S(1); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 1", [&]{ Vec b; Random r = S(1); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 2", [&]{ Vec b; Random r = S(2); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 2", [&]{ Vec b; Random r = S(2); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 3", [&]{ Vec b; Random r = S(3); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 3", [&]{ Vec b; Random r = S(3); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 4", [&]{ Vec b; Random r = S(4); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 4", [&]{ Vec b; Random r = S(4); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 5", [&]{ Vec b; Random r = S(5); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 5", [&]{ Vec b; Random r = S(5); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 6", [&]{ Vec b; Random r = S(6); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 6", [&]{ Vec b; Random r = S(6); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 7", [&]{ Vec b; Random r = S(7); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 7", [&]{ Vec b; Random r = S(7); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 8", [&]{ Vec b; Random r = S(8); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 8", [&]{ Vec b; Random r = S(8); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 9", [&]{ Vec b; Random r = S(9); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 9", [&]{ Vec b; Random r = S(9); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.1 seed 10", [&]{ Vec b; Random r = S(10); return GraspTSP(d20, b, 0.1, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.1 seed 10", [&]{ Vec b; Random r = S(10); return GraspKP(p20, w20, c20, b, 0.1, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 1", [&]{ Vec b; Random r = S(1); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 1", [&]{ Vec b; Random r = S(1); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 2", [&]{ Vec b; Random r = S(2); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 2", [&]{ Vec b; Random r = S(2); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 3", [&]{ Vec b; Random r = S(3); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 3", [&]{ Vec b; Random r = S(3); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 4", [&]{ Vec b; Random r = S(4); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 4", [&]{ Vec b; Random r = S(4); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 5", [&]{ Vec b; Random r = S(5); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 5", [&]{ Vec b; Random r = S(5); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 6", [&]{ Vec b; Random r = S(6); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 6", [&]{ Vec b; Random r = S(6); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 7", [&]{ Vec b; Random r = S(7); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 7", [&]{ Vec b; Random r = S(7); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 8", [&]{ Vec b; Random r = S(8); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 8", [&]{ Vec b; Random r = S(8); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 9", [&]{ Vec b; Random r = S(9); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 9", [&]{ Vec b; Random r = S(9); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.25 seed 10", [&]{ Vec b; Random r = S(10); return GraspTSP(d20, b, 0.25, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.25 seed 10", [&]{ Vec b; Random r = S(10); return GraspKP(p20, w20, c20, b, 0.25, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 1", [&]{ Vec b; Random r = S(1); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 1", [&]{ Vec b; Random r = S(1); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 2", [&]{ Vec b; Random r = S(2); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 2", [&]{ Vec b; Random r = S(2); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 3", [&]{ Vec b; Random r = S(3); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 3", [&]{ Vec b; Random r = S(3); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 4", [&]{ Vec b; Random r = S(4); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 4", [&]{ Vec b; Random r = S(4); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 5", [&]{ Vec b; Random r = S(5); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 5", [&]{ Vec b; Random r = S(5); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 6", [&]{ Vec b; Random r = S(6); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 6", [&]{ Vec b; Random r = S(6); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 7", [&]{ Vec b; Random r = S(7); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 7", [&]{ Vec b; Random r = S(7); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 8", [&]{ Vec b; Random r = S(8); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 8", [&]{ Vec b; Random r = S(8); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 9", [&]{ Vec b; Random r = S(9); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 9", [&]{ Vec b; Random r = S(9); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=0.5 seed 10", [&]{ Vec b; Random r = S(10); return GraspTSP(d20, b, 0.5, 20, r); }(), 291);
    check("KP20 GRASP alpha=0.5 seed 10", [&]{ Vec b; Random r = S(10); return GraspKP(p20, w20, c20, b, 0.5, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 1", [&]{ Vec b; Random r = S(1); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 1", [&]{ Vec b; Random r = S(1); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 2", [&]{ Vec b; Random r = S(2); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 2", [&]{ Vec b; Random r = S(2); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 519);
    check("TSP20 GRASP alpha=1 seed 3", [&]{ Vec b; Random r = S(3); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 3", [&]{ Vec b; Random r = S(3); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 4", [&]{ Vec b; Random r = S(4); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 4", [&]{ Vec b; Random r = S(4); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 5", [&]{ Vec b; Random r = S(5); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 5", [&]{ Vec b; Random r = S(5); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 6", [&]{ Vec b; Random r = S(6); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 6", [&]{ Vec b; Random r = S(6); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 7", [&]{ Vec b; Random r = S(7); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 7", [&]{ Vec b; Random r = S(7); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 8", [&]{ Vec b; Random r = S(8); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 8", [&]{ Vec b; Random r = S(8); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 9", [&]{ Vec b; Random r = S(9); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 9", [&]{ Vec b; Random r = S(9); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 GRASP alpha=1 seed 10", [&]{ Vec b; Random r = S(10); return GraspTSP(d20, b, 1.0, 20, r); }(), 291);
    check("KP20 GRASP alpha=1 seed 10", [&]{ Vec b; Random r = S(10); return GraspKP(p20, w20, c20, b, 1.0, 20, r); }(), 521);
    check("TSP20 LNS q=2 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=2 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return LnsTSP(d20, pi, 2, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=3 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return LnsTSP(d20, pi, 3, 200, r); }(), 319);
    check("TSP20 LNS q=5 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return LnsTSP(d20, pi, 5, 200, r); }(), 319);
    check("TSP20 LNS q=5 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=5 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return LnsTSP(d20, pi, 5, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 1", [&]{ Vec pi = nn20; Random r = S(1); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 2", [&]{ Vec pi = nn20; Random r = S(2); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 3", [&]{ Vec pi = nn20; Random r = S(3); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 4", [&]{ Vec pi = nn20; Random r = S(4); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 5", [&]{ Vec pi = nn20; Random r = S(5); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 6", [&]{ Vec pi = nn20; Random r = S(6); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 7", [&]{ Vec pi = nn20; Random r = S(7); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 8", [&]{ Vec pi = nn20; Random r = S(8); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 9", [&]{ Vec pi = nn20; Random r = S(9); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 LNS q=8 seed 10", [&]{ Vec pi = nn20; Random r = S(10); return LnsTSP(d20, pi, 8, 200, r); }(), 291);
    check("TSP20 ALNS seed 1", [&]{ Vec pi = nn20; Random r = S(1); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 1 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(1); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 846);
    check("TSP20 ALNS seed 2", [&]{ Vec pi = nn20; Random r = S(2); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 2 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(2); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 971);
    check("TSP20 ALNS seed 3", [&]{ Vec pi = nn20; Random r = S(3); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 3 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(3); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 768);
    check("TSP20 ALNS seed 4", [&]{ Vec pi = nn20; Random r = S(4); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 4 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(4); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 803);
    check("TSP20 ALNS seed 5", [&]{ Vec pi = nn20; Random r = S(5); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 319);
    check("TSP20 ALNS seed 5 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(5); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 839);
    check("TSP20 ALNS seed 6", [&]{ Vec pi = nn20; Random r = S(6); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 319);
    check("TSP20 ALNS seed 6 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(6); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 980);
    check("TSP20 ALNS seed 7", [&]{ Vec pi = nn20; Random r = S(7); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 7 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(7); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 946);
    check("TSP20 ALNS seed 8", [&]{ Vec pi = nn20; Random r = S(8); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 8 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(8); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 827);
    check("TSP20 ALNS seed 9", [&]{ Vec pi = nn20; Random r = S(9); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 291);
    check("TSP20 ALNS seed 9 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(9); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 912);
    check("TSP20 ALNS seed 10", [&]{ Vec pi = nn20; Random r = S(10); double wg[2]; return AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); }(), 319);
    check("TSP20 ALNS seed 10 weight1 x1000", [&]{ Vec pi = nn20; Random r = S(10); double wg[2]; AlnsTSP(d20, pi, 5, 200, 0.1, r, wg); return (int) (1000 * wg[0]); }(), 793);
    check("KP20 GA seed 1", [&]{ Vec b; Random r = S(1); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 521);
    check("KP20 GA seed 2", [&]{ Vec b; Random r = S(2); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 519);
    check("KP20 GA seed 3", [&]{ Vec b; Random r = S(3); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 521);
    check("KP20 GA seed 4", [&]{ Vec b; Random r = S(4); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 508);
    check("KP20 GA seed 5", [&]{ Vec b; Random r = S(5); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 514);
    check("KP20 GA seed 6", [&]{ Vec b; Random r = S(6); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 521);
    check("KP20 GA seed 7", [&]{ Vec b; Random r = S(7); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 510);
    check("KP20 GA seed 8", [&]{ Vec b; Random r = S(8); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 519);
    check("KP20 GA seed 9", [&]{ Vec b; Random r = S(9); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 521);
    check("KP20 GA seed 10", [&]{ Vec b; Random r = S(10); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, false, r); }(), 515);
    check("KP20 memetic seed 1", [&]{ Vec b; Random r = S(1); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 2", [&]{ Vec b; Random r = S(2); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 3", [&]{ Vec b; Random r = S(3); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 4", [&]{ Vec b; Random r = S(4); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 5", [&]{ Vec b; Random r = S(5); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 6", [&]{ Vec b; Random r = S(6); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 7", [&]{ Vec b; Random r = S(7); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 8", [&]{ Vec b; Random r = S(8); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 9", [&]{ Vec b; Random r = S(9); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("KP20 memetic seed 10", [&]{ Vec b; Random r = S(10); return GeneticKP(p20, w20, c20, b, 10, 0.05, 300, true, r); }(), 521);
    check("PCM15 LPT", [&]{ Vec L = Loads(LPT(t15, m15), t15, m15); int C = 0; for (int x : L) if (x > C) C = x; return C; }(), 73);
    check("PCM15 TS tenure=0", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 0, 300); }(), 73);
    check("PCM15 TS tenure=1", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 1, 300); }(), 73);
    check("PCM15 TS tenure=2", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 2, 300); }(), 73);
    check("PCM15 TS tenure=3", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 3, 300); }(), 73);
    check("PCM15 TS tenure=5", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 5, 300); }(), 73);
    check("PCM15 TS tenure=7", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 7, 300); }(), 72);
    check("PCM15 TS tenure=10", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 10, 300); }(), 71);
    check("PCM15 TS tenure=15", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 15, 300); }(), 69);
    check("PCM15 TS tenure=20", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 20, 300); }(), 70);
    check("PCM15 TS tenure=30", [&]{ Vec mach = LPT(t15, m15); return TabuSearchPCmax(t15, m15, mach, 30, 300); }(), 70);
    check("PCM15 VNS kmax=1 seed 1", [&]{ Vec mach = LPT(t15, m15); Random r = S(1); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=1 seed 2", [&]{ Vec mach = LPT(t15, m15); Random r = S(2); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 70);
    check("PCM15 VNS kmax=1 seed 3", [&]{ Vec mach = LPT(t15, m15); Random r = S(3); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 70);
    check("PCM15 VNS kmax=1 seed 4", [&]{ Vec mach = LPT(t15, m15); Random r = S(4); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 70);
    check("PCM15 VNS kmax=1 seed 5", [&]{ Vec mach = LPT(t15, m15); Random r = S(5); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=1 seed 6", [&]{ Vec mach = LPT(t15, m15); Random r = S(6); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=1 seed 7", [&]{ Vec mach = LPT(t15, m15); Random r = S(7); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=1 seed 8", [&]{ Vec mach = LPT(t15, m15); Random r = S(8); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=1 seed 9", [&]{ Vec mach = LPT(t15, m15); Random r = S(9); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 70);
    check("PCM15 VNS kmax=1 seed 10", [&]{ Vec mach = LPT(t15, m15); Random r = S(10); return VNSPCmax(t15, m15, mach, 1, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 1", [&]{ Vec mach = LPT(t15, m15); Random r = S(1); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 70);
    check("PCM15 VNS kmax=2 seed 2", [&]{ Vec mach = LPT(t15, m15); Random r = S(2); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 3", [&]{ Vec mach = LPT(t15, m15); Random r = S(3); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 4", [&]{ Vec mach = LPT(t15, m15); Random r = S(4); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 5", [&]{ Vec mach = LPT(t15, m15); Random r = S(5); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 6", [&]{ Vec mach = LPT(t15, m15); Random r = S(6); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 70);
    check("PCM15 VNS kmax=2 seed 7", [&]{ Vec mach = LPT(t15, m15); Random r = S(7); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=2 seed 8", [&]{ Vec mach = LPT(t15, m15); Random r = S(8); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 70);
    check("PCM15 VNS kmax=2 seed 9", [&]{ Vec mach = LPT(t15, m15); Random r = S(9); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 70);
    check("PCM15 VNS kmax=2 seed 10", [&]{ Vec mach = LPT(t15, m15); Random r = S(10); return VNSPCmax(t15, m15, mach, 2, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 1", [&]{ Vec mach = LPT(t15, m15); Random r = S(1); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 70);
    check("PCM15 VNS kmax=3 seed 2", [&]{ Vec mach = LPT(t15, m15); Random r = S(2); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 3", [&]{ Vec mach = LPT(t15, m15); Random r = S(3); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 4", [&]{ Vec mach = LPT(t15, m15); Random r = S(4); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 5", [&]{ Vec mach = LPT(t15, m15); Random r = S(5); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 6", [&]{ Vec mach = LPT(t15, m15); Random r = S(6); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 7", [&]{ Vec mach = LPT(t15, m15); Random r = S(7); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("PCM15 VNS kmax=3 seed 8", [&]{ Vec mach = LPT(t15, m15); Random r = S(8); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 70);
    check("PCM15 VNS kmax=3 seed 9", [&]{ Vec mach = LPT(t15, m15); Random r = S(9); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 70);
    check("PCM15 VNS kmax=3 seed 10", [&]{ Vec mach = LPT(t15, m15); Random r = S(10); return VNSPCmax(t15, m15, mach, 3, 300, r); }(), 69);
    check("BPP24 FFD", [&]{ Vec bin; return FirstFit(w24, c24, ord24, bin); }(), 10);
    check("BPP24 LNS q=2 FFD seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 FFD seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 2, false, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=2 exact seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 2, true, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 FFD seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 3, false, 200, r); }(), 10);
    check("BPP24 LNS q=3 exact seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=3 exact seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 3, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 FFD seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 FFD seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 4, false, 200, r); }(), 10);
    check("BPP24 LNS q=4 exact seed 1", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(1); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 2", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(2); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 3", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(3); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 4", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(4); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 5", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(5); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 6", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(6); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 7", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(7); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 8", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(8); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 9", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(9); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("BPP24 LNS q=4 exact seed 10", [&]{ Vec bin; int nb = FirstFit(w24, c24, ord24, bin); Random r = S(10); return LnsBPP(w24, c24, bin, nb, 4, true, 200, r); }(), 9);
    check("EX_SA_1", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; Random r = G(vector<double>{0.13,0.76,0.46,0.53,0.22,0.05,0.68,0.68,0.93,0.38,0.52,0.83}); return SimulatedAnnealingTSP(d, pi, 5.0, 0.8, 2, 3.0, r); }(), 20);
    check("EX_SA_2 beta=1", [&]{ Vec x = Vec{1,1,0,1,0}; Random r = G(vector<double>{0.63,0.11,0.17,0.66,0.38,0.94,0.58,0.59,0.69,0.67,0.39,0.62,0.69,0.07,0.59,0.42}); return SimulatedAnnealingKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, x, 1, 4.0, 0.8, 2, 1.7, r); }(), 24);
    check("EX_SA_2 beta=3", [&]{ Vec x = Vec{1,1,0,1,0}; Random r = G(vector<double>{0.63,0.11,0.17,0.66,0.38,0.94,0.58,0.59,0.69,0.67,0.39,0.62,0.69,0.07,0.59,0.42}); return SimulatedAnnealingKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, x, 3, 4.0, 0.8, 2, 1.7, r); }(), 29);
    check("EX_SA_3", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; Random r = G(vector<double>{0.13,0.76,0.46,0.53,0.22,0.05,0.68,0.68,0.93,0.38,0.52,0.83}); return SimulatedAnnealingTSP(d, pi, 0.5, 0.8, 2, 0.3, r); }(), 22);
    check("EX_TS_1", [&]{ Vec t = Vec{3,3,2,2,2}; Vec mach = LPT(t, 2); return TabuSearchPCmax(t, 2, mach, 3, 20); }(), 6);
    check("EX_TS_2 tenure=0", [&]{ Vec t = Vec{3,3,2,2,2}; Vec mach = LPT(t, 2); return TabuSearchPCmax(t, 2, mach, 0, 50); }(), 7);
    check("EX_TS_2 tenure=1", [&]{ Vec t = Vec{3,3,2,2,2}; Vec mach = LPT(t, 2); return TabuSearchPCmax(t, 2, mach, 1, 50); }(), 7);
    check("EX_TS_3", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; return TabuSearchTSP(d, pi, 2, 3); }(), 20);
    check("EX_IV_1", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; Random r = G(vector<double>{0.13,0.76,0.46}); return IteratedLocalSearchTSP(d, pi, 1, r); }(), 20);
    check("EX_IV_2", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; Random r = G(vector<double>{0.16,0.13,0.01}); return IteratedLocalSearchTSP(d, pi, 1, r); }(), 22);
    check("EX_IV_3", [&]{ Vec t = Vec{4,4,3,3,2,2,2}; Vec mach = LPT(t, 3); Random r = G(vector<double>{0.26,0.51,0.92,0.07,0.44,0.09,0.36,0.36,0.87,0.77,0.04,0.66}); return VNSPCmax(t, 3, mach, 3, 10, r); }(), 7);
    check("EX_GRASP_1", [&]{ Vec b; Random r = G(vector<double>{0.39,0.27,0.38,0.6,0.66,0.14,0.04,0.04,0.8,0.15}); return GraspKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, b, 0.5, 2, r); }(), 29);
    check("EX_GRASP_2", [&]{ Vec b; Random r = G(vector<double>{0.45,0.31,0.05,0.86,0.41,0.52,0.47,0.47,0.28,0.22}); return GraspTSP(Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}, b, 0.5, 2, r); }(), 20);
    check("EX_GRASP_3 alpha=0", [&]{ Vec b; Random r = G(vector<double>{0.92,0.29,0.21,0.73,0.53,0.33,0.75,0.76,0.54,0.68,0.64,0.82}); return GraspKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, b, 0.0, 2, r); }(), 24);
    check("EX_GRASP_3 alpha=1", [&]{ Vec b; Random r = G(vector<double>{0.92,0.29,0.21,0.73,0.53,0.33,0.75,0.76,0.54,0.68,0.64,0.82}); return GraspKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, b, 1.0, 2, r); }(), 24);
    check("EX_LNS_1", [&]{ Vec w = Vec{2,7,5,5,3,2,6,6}; Vec bin; int nb = FirstFit(w, 10, Vec{0,1,2,3,4,5,6,7}, bin); Random r = G(vector<double>{0.13,0.76,0.46,0.53,0.22,0.05,0.68,0.68,0.93,0.38}); return LnsBPP(w, 10, bin, nb, 2, false, 10, r); }(), 4);
    check("EX_LNS_2 exact", [&]{ Vec w = Vec{3,3,6,5,4,4,2,3}; Vec bin; int nb = FirstFit(w, 10, Vec{2,3,4,5,0,1,7,6}, bin); Random r = G(vector<double>{0.66,0.78,0.29,0.66}); return LnsBPP(w, 10, bin, nb, 3, true, 1, r); }(), 3);
    check("EX_LNS_2 FFD", [&]{ Vec w = Vec{3,3,6,5,4,4,2,3}; Vec bin; int nb = FirstFit(w, 10, Vec{2,3,4,5,0,1,7,6}, bin); Random r = G(vector<double>{0.66,0.78,0.29,0.66}); return LnsBPP(w, 10, bin, nb, 3, false, 1, r); }(), 4);
    check("EX_LNS_3", [&]{ Matrix d = Matrix{Vec{0,5,1,7,7,4},Vec{5,0,4,5,2,1},Vec{1,4,0,6,6,4},Vec{7,5,6,0,7,6},Vec{7,2,6,7,0,2},Vec{4,1,4,6,2,0}}; Vec pi = Vec{0,2,1,5,4,3}; Random r = G(vector<double>{0.68,0.16,0.84,0.92}); return LnsTSP(d, pi, 2, 2, r); }(), 20);
    check("EX_GA_1", [&]{ Vec b; Random r = G(vector<double>{0.75,0.75,0.75,0.25,0.25,0.75,0.25,0.75,0.75,0.25,0.75,0.25,0.75,0.25,0.25,0.25,0.25,0.75,0.25,0.75,0.58,0.07,0.5,0.39,0.63,0.56,0.15,0.15,0.22,0.6,0.23,0.97,0.41,0.64,0.36,0.05,0.09,0.6,0.8,0.01}); return GeneticKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, b, 4, 0.2, 2, false, r); }(), 24);
    check("EX_GA_2", [&]{ Vec b; Random r = G(vector<double>{0.75,0.75,0.75,0.25,0.25,0.75,0.25,0.75,0.75,0.25,0.75,0.25,0.75,0.25,0.25,0.25,0.25,0.75,0.25,0.75,0.58,0.07,0.5,0.39,0.63,0.56,0.15,0.15,0.22,0.6,0.23,0.97,0.41,0.64,0.36,0.05,0.09,0.6,0.8,0.01}); return GeneticKP(Vec{11,5,18,8,8}, Vec{5,3,11,6,7}, 16, b, 4, 0.2, 2, true, r); }(), 29);
    printf("\n%d checks, %d errors\n", checks, errors);
    return errors > 0 ? 1 : 0;
}

Writing src/Main.cpp


In [41]:
!g++ -std=c++17 -O2 -o METAHEUR src/*.cpp && ./METAHEUR

TSP ref: nearest neighbour                         25 OK
Ex SA TSP ref (seed 1)                             18 OK
Ex SA KP ref (seed 3)                              17 OK
Ex TS PCmax ref tenure=0                           11 OK
Ex TS PCmax ref tenure=1                           11 OK
Ex TS PCmax ref tenure=2                           11 OK
Ex TS PCmax ref tenure=3                           11 OK
Ex TS PCmax ref tenure=4                           10 OK
Ex TS PCmax ref tenure=5                           10 OK
Ex TS PCmax ref tenure=6                            9 OK
Ex VNS PCmax ref (seed 1)                           9 OK
Ex GRASP KP ref (seed 2)                           17 OK
Ex LNS TSP ref (seed 1)                            18 OK
Ex LNS BPP ref (seed 3)                             3 OK
Ex GA KP ref (seed 36)                             17 OK
TSP20 nearest neighbour                           326 OK
TSP20 2-opt                                       319 OK
TSP20 SA alpha=0.9 T0=1 seed 1 

## Try it yourself

Change the data in the cell below and run it again.

In [42]:
%%writefile try.cpp
#include <cstdio>
#include "src/Metaheuristics.h"

// Try it yourself: change the data below, compile and run again
Matrix d = Matrix{Vec{0,5,1,7,7,4}, Vec{5,0,4,5,2,1}, Vec{1,4,0,6,6,4},
                  Vec{7,5,6,0,7,6}, Vec{7,2,6,7,0,2}, Vec{4,1,4,6,2,0}};   // TSP: symmetric distances
Vec p = Vec{11,5,18,8,8}, w = Vec{5,3,11,6,7}; int c = 16;               // KP: profits, weights, capacity
long long seed = 1;                                                      // seed of the random numbers

static Random R() { Random r; r.seed = seed; r.used = 0; return r; }

static void Show(const char* name, int value, const Vec& sol)
{
    printf("%-22s %3d  [", name, value);
    for (int k = 0; k < (int) sol.size(); ++k) printf(k > 0 ? " %d" : "%d", sol[k]);
    printf("]\n");
}

int main()
{
    Random r;
    Vec nn = NearestNeighbour(d), pi, x;
    Show("TSP nearest neighbour", TourLength(nn, d), nn);
    pi = nn; r = R(); Show("TSP simulated annealing", SimulatedAnnealingTSP(d, pi, 10.0, 0.9, 20, 0.1, r), pi);
    pi = nn; Show("TSP tabu search", TabuSearchTSP(d, pi, 2, 50), pi);
    pi = nn; r = R(); Show("TSP iterated LS", IteratedLocalSearchTSP(d, pi, 20, r), pi);
    r = R(); int len = GraspTSP(d, pi, 0.3, 10, r); Show("TSP GRASP", len, pi);
    pi = nn; r = R(); Show("TSP LNS", LnsTSP(d, pi, 2, 50, r), pi);
    x = Vec(p.size(), 0); r = R(); Show("KP simulated annealing", SimulatedAnnealingKP(p, w, c, x, 3, 10.0, 0.9, 20, 0.1, r), x);
    r = R(); int z = GraspKP(p, w, c, x, 0.3, 10, r); Show("KP GRASP", z, x);
    r = R(); z = GeneticKP(p, w, c, x, 6, 0.1, 50, false, r); Show("KP genetic", z, x);
    r = R(); z = GeneticKP(p, w, c, x, 6, 0.1, 50, true, r); Show("KP memetic", z, x);
    return 0;
}

Writing try.cpp


In [43]:
!g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp) && ./try

TSP nearest neighbour   22  [0 2 1 5 4 3]
TSP simulated annealing  20  [0 5 4 1 3 2]
TSP tabu search         20  [0 2 3 1 4 5]
TSP iterated LS         20  [0 2 3 1 4 5]
TSP GRASP               20  [0 2 3 1 4 5]
TSP LNS                 20  [0 5 4 1 3 2]
KP simulated annealing  29  [1 0 1 0 0]
KP GRASP                29  [1 0 1 0 0]
KP genetic              29  [1 0 1 0 0]
KP memetic              29  [1 0 1 0 0]


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).